# **Dados e Aprendizagem Automática** 

## **Decision Trees - Tratamento 2**
Nesta segunda tentativa, no nosso processo de "Data Preparation" começámos por converter a coluna 'RECORD_DATE' em características temporais acionáveis, extraindo especificamente a hora, o dia da semana e o mês, ao mesmo tempo que se criaram indicadores binários para fins de semana e horas de ponta de modo a captar os ciclos de tráfego, sendo a coluna de data original removida posteriormente. Para abordar questões de qualidade dos dados, normalizámos as entradas de texto inconsistentes nas colunas 'AVERAGE_CLOUDINESS' e 'AVERAGE_RAIN' (corrigindo erros de codificação e agrupando sinónimos) e demos _input_ dos valores em falta criando categorias explícitas de "no_rain" e "unknown_cloudiness" em vez de eliminar linhas. De seguida, removemos a coluna 'CITY_NAME' por conter um valor constante ("Porto"), bem como a coluna 'AVERAGE_PRECIPITATION', dado que consistia quase apenas em zeros. Por fim, aplicámos One-Hot Encoding às variáveis categóricas ('LUMINOSITY', 'AVERAGE_CLOUDINESS' e 'AVERAGE_RAIN') para as transformar num formato legível por máquina sem impor uma ordem artificial.

**Imports necessários para este teste**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report

%matplotlib inline

### **Preparation**
**Load the CSVs**

In [ ]:
df_train = pd.read_csv('../../Datasets/training_data.csv', encoding='latin-1')
df_test = pd.read_csv('../../Datasets/test_data.csv', encoding='latin-1')

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (6812, 14)
Test shape: (1500, 13)


**Feature Engineering**

In [55]:
def extract_date_features(df):
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek # Monday=0, Sunday=6
    df['month'] = df['record_date'].dt.month
    
    # Create "Weekend" feature
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Create "Rush Hour" feature (7 da manhã até às 9 da manhã e 4 da tarde ate às 7 da tarde, podemos brincar com estas horas)
    df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)
    
    return df.drop(columns=['record_date'])

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)


**Missing Values e Valores Incorretos**

In [56]:
def clean_categorical_text(df):

    # Primeiro "limpamos" a coluna 'AVERAGE CLOUDINESS'
    correcoes_erros = {
        'cï¿½u': 'ceu',      # erro especifico
        'c\u00e9u': 'ceu', # é
        'c\u00fa': 'ceu',  # ú
        'c\u00fau': 'ceu', 
        'céu': 'ceu'
    }
    # regex=True permite substituir apenas parte da frase (ex: "cï¿½u claro" -> "ceu claro")
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(correcoes_erros, regex=True)

    cloud_map = {
        'ceu claro': 'ceu_claro',
        'ceu limpo': 'ceu_claro',

        'ceu pouco nublado': 'pouco_nublado',
        'nuvens dispersas': 'pouco_nublado',
        'algumas nuvens': 'pouco_nublado',

        'nuvens quebrados': 'nublado', 
        'nuvens quebradas': 'nublado',
        'tempo nublado': 'nublado',
        'nublado': 'nublado',
    }
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(cloud_map, regex=True)
    # Tratamos também dos seus missing values
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('nan', 'unknown_cloudiness')
    
    # Depois "limpamos" também a coluna da "AVERAGE RAIN"
    rain_map = {
        'chuva fraca': 'chuva_fraca',
        'chuva leve': 'chuva_fraca',
        'aguaceiros fracos': 'chuva_fraca',
        'chuvisco fraco': 'chuva_fraca',
        'chuvisco e chuva fraca': 'chuva_fraca',
        'trovoada com chuva leve': 'chuva_fraca', 

        'chuva moderada': 'chuva_moderada',
        'chuva': 'chuva_moderada',
        'aguaceiros': 'chuva_moderada',
        'trovoada com chuva': 'chuva_moderada',

        'chuva forte': 'chuva_forte',
        'chuva de intensidade pesada': 'chuva_forte',
        'chuva de intensidade pesado': 'chuva_forte'
    }
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].replace(rain_map)
    # Tratamos também dos seus missing values
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].fillna('no_rain')
    
    #df["LUMINOSITY"] = df_train["LUMINOSITY"].replace("LOW_LIGHT", "LIGHT")
    
    return df

df_train = clean_categorical_text(df_train)
df_test = clean_categorical_text(df_test)

**Verificação dos valores dessas colunas agora**

In [57]:
df_test['AVERAGE_CLOUDINESS'].value_counts()

AVERAGE_CLOUDINESS
unknown_cloudiness    599
ceu_claro             372
pouco_nublado         301
nublado               228
Name: count, dtype: int64

In [58]:
df_test['AVERAGE_RAIN'].value_counts()

AVERAGE_RAIN
no_rain           1360
chuva_fraca         93
chuva_moderada      47
Name: count, dtype: int64

**Drop de Colunas Redundates** 

Considera-mo-las redundantes devido a 'CITY_NAME' conter um valor constante ("Porto") e 'AVERAGE_PRECIPITATION' consistir quase apenas em zeros.

In [59]:
cols_to_drop = ['city_name', 'AVERAGE_PRECIPITATION']
df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

**Handling categoric data**

In [60]:
categorical_features = ['LUMINOSITY', 'AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']

# Inicializar o Encoder
# handle_unknown='ignore' é CRUCIAL: Se o teste tiver uma categoria nova que o treino não conhece,
# ele ignora (mete zeros) em vez de dar erro.
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit APENAS no Treino
# O encoder "aprende" apenas as colunas que existem no treino
encoder.fit(df_train[categorical_features])

# Transformar ambos
# Aplicamos a estrutura que aprendemos no treino aos dois datasets
train_encoded = pd.DataFrame(encoder.transform(df_train[categorical_features]), 
                             columns=encoder.get_feature_names_out())

test_encoded = pd.DataFrame(encoder.transform(df_test[categorical_features]), 
                            columns=encoder.get_feature_names_out())

# Reset Index (Importante para o concat não falhar)
df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)
train_encoded.reset_index(drop=True, inplace=True)
test_encoded.reset_index(drop=True, inplace=True)

# Juntar tudo e remover as colunas originais
df_train = pd.concat([df_train.drop(columns=categorical_features), train_encoded], axis=1)
df_test = pd.concat([df_test.drop(columns=categorical_features), test_encoded], axis=1)

print("New Train shape:", df_train.shape)
print("New Test shape:", df_test.shape)

New Train shape: (6812, 24)
New Test shape: (1500, 23)


In [61]:
# Corrigir missing values do de treino
df_train["AVERAGE_SPEED_DIFF"] = df_train["AVERAGE_SPEED_DIFF"].fillna("None")
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6812 entries, 0 to 6811
Data columns (total 24 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   AVERAGE_SPEED_DIFF                     6812 non-null   object 
 1   AVERAGE_FREE_FLOW_SPEED                6812 non-null   float64
 2   AVERAGE_TIME_DIFF                      6812 non-null   float64
 3   AVERAGE_FREE_FLOW_TIME                 6812 non-null   float64
 4   AVERAGE_TEMPERATURE                    6812 non-null   float64
 5   AVERAGE_ATMOSP_PRESSURE                6812 non-null   float64
 6   AVERAGE_HUMIDITY                       6812 non-null   float64
 7   AVERAGE_WIND_SPEED                     6812 non-null   float64
 8   hour                                   6812 non-null   int32  
 9   day_of_week                            6812 non-null   int32  
 10  month                                  6812 non-null   int32  
 11  is_w

In [62]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 23 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   AVERAGE_FREE_FLOW_SPEED                1500 non-null   float64
 1   AVERAGE_TIME_DIFF                      1500 non-null   float64
 2   AVERAGE_FREE_FLOW_TIME                 1500 non-null   float64
 3   AVERAGE_TEMPERATURE                    1500 non-null   float64
 4   AVERAGE_ATMOSP_PRESSURE                1500 non-null   float64
 5   AVERAGE_HUMIDITY                       1500 non-null   float64
 6   AVERAGE_WIND_SPEED                     1500 non-null   float64
 7   hour                                   1500 non-null   int32  
 8   day_of_week                            1500 non-null   int32  
 9   month                                  1500 non-null   int32  
 10  is_weekend                             1500 non-null   int64  
 11  is_r

### **Modeling**

Select features and target

In [63]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

In [64]:
X_train = df_train.drop(columns=["AVERAGE_SPEED_DIFF"])
y_train = df_train["AVERAGE_SPEED_DIFF"]

In [65]:
X_test = df_test

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

**Validation Block**

In [66]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Cópias
X_val_copy = X_train.copy()
y_val_copy = y_train.copy()

# criar conjunto de val
X_tr, X_val, y_tr, y_val = train_test_split(
    X_val_copy, y_val_copy, test_size=0.2, random_state=2022, stratify=y_val_copy
)

# Treinar modelo só para validar
clf_val = DecisionTreeClassifier(random_state=2022)
clf_val.fit(X_tr, y_tr)

# Prever na validação
val_preds = clf_val.predict(X_val)

# Mostrar accuracy
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:\n", classification_report(y_val, val_preds))


Validation Accuracy: 0.7329420396184886

Classification Report:
               precision    recall  f1-score   support

        High       0.67      0.70      0.68       213
         Low       0.62      0.62      0.62       284
      Medium       0.69      0.66      0.68       330
        None       0.86      0.87      0.86       440
   Very_High       0.77      0.77      0.77        96

    accuracy                           0.73      1363
   macro avg       0.72      0.72      0.72      1363
weighted avg       0.73      0.73      0.73      1363



**Predict**

Predict on test set

In [67]:
clf = DecisionTreeClassifier(random_state=2022)
clf.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,2022
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [68]:
preds = clf.predict(X_test)

Create csv

In [ ]:
submission = pd.DataFrame({
    "RowId": range(1, len(preds)+1),
    "Speed_Diff": preds
})

submission.to_csv("../../results/DecisionTrees/dt2.csv", index=False)
submission.head()

,RowId,Speed_Diff
0,1,None
1,2,Medium
2,3,None
3,4,High
4,5,None
